##Install Dependencies

In [ ]:
!pip install -q pandas scikit-learn langchain-openai nest_asyncio datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 8.2 MB/s eta 0:00:00


##Setup Models & API Keys

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

# Set up  OpenRouter Credentials
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

print("Initializing the 6 Competitor Judges via OpenRouter...")

judges = {
    "GPT-4O-MINI": ChatOpenAI(model="openai/gpt-4o-mini", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0),
    "MISTRAL-LARGE": ChatOpenAI(model="mistralai/mistral-large", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0),
    "GEMINI-FLASH": ChatOpenAI(model="google/gemini-2.5-flash", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0),
    "CLAUDE-SONNET": ChatOpenAI(model="anthropic/claude-sonnet-4.6", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0),
    "LLAMA-3.1-70B": ChatOpenAI(model="meta-llama/llama-3.1-70b-instruct", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0),
    "DEEPSEEK-CHAT": ChatOpenAI(model="deepseek/deepseek-chat", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0)
}

print("All 6 judges are online and ready to grade!")

Initializing the 6 Competitor Judges via OpenRouter...
All 6 judges are online and ready to grade!


##Fetch the Peer-Reviewed Dataset (HaluEval)

In [ ]:
import pandas as pd
from datasets import load_dataset

print("Pulling the HaluEval QA Evaluation Dataset from HuggingFace...")

# Load the official Question Answering split of HaluEval
raw_dataset = load_dataset("pminervini/HaluEval", name="qa", split="data")
full_df = pd.DataFrame(raw_dataset)

# Scale up to 100 'Right' (Faithful) pairs and 100 'Hallucinated' pairs
sample_size = 100
faithful_samples = full_df.sample(n=sample_size, random_state=42).copy()
hallucinated_samples = full_df.sample(n=sample_size, random_state=24).copy()

eval_rows = []

# Add the faithful cases (Ground Truth = 1)
for _, row in faithful_samples.iterrows():
    eval_rows.append({
        "context": row['knowledge'],
        "answer": row['right_answer'],
        "ground_truth": 1
    })

# Add the hallucinated cases (Ground Truth = 0)
for _, row in hallucinated_samples.iterrows():
    eval_rows.append({
        "context": row['knowledge'],
        "answer": row['hallucinated_answer'],
        "ground_truth": 0
    })

# Shuffle the final dataset so there is zero position pattern bias
df_eval = pd.DataFrame(eval_rows).sample(frac=1, random_state=99).reset_index(drop=True)

print(f"Replicated Benchmark Dataset Created! Loaded {len(df_eval)} peer-reviewed evaluation rows.")
print("Distribution: 100 Faithful Answers (1) | 100 Hallucinated Answers (0)")

Pulling the HaluEval QA Evaluation Dataset from HuggingFace...


README.md:   0%|          | 0.00/2.88k [00:00<?, ?B/s]

qa/data-00000-of-00001.parquet:   0%|          | 0.00/3.75M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Replicated Benchmark Dataset Created! Loaded 200 peer-reviewed evaluation rows.
Distribution: 100 Faithful Answers (1) | 100 Hallucinated Answers (0)


## Define the Evaluation Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate

judge_prompt = PromptTemplate(
    template="""You are an expert, highly objective AI data annotator. Your sole job is to evaluate the 'Faithfulness' of an Answer based strictly on a given Context.

RULES:
- Output ONLY the number 1 if the answer is completely Faithful.
- Output ONLY the number 0 if the answer contains a Hallucination.
- "Faithful" (1) means all facts stated in the answer are directly supported by the text. Missing details are acceptable.
- "Hallucination" (0) means the answer introduces facts, speculations, or numbers not present in the context.

Context:
{context}

Answer:
{answer}

Score (1 or 0):""",
    input_variables=["context", "answer"]
)
print("Strictly bound binary evaluation prompt loaded.")

Strictly bound binary evaluation prompt loaded.


## The 6-Model Benchmark Loop

In [ ]:
import time

# Dynamic results tracker dict based on your setup keys
results = {model_name: [] for model_name in judges.keys()}

print("RUNNING THE 6-MODEL BENCHMARK LOOP (200 TOTAL CASES)...")

for index, row in df_eval.iterrows():
    formatted_prompt = judge_prompt.format(context=row['context'], answer=row['answer'])

    # Run every single model on this exact row
    for model_name, model_instance in judges.items():
        try:
            raw_out = model_instance.invoke(formatted_prompt).content.strip()
            # Clean response text down to a strict integer parse
            clean_score = int(raw_out) if raw_out in ["0", "1"] else 0
            results[model_name].append(clean_score)
        except Exception as e:
            results[model_name].append(0) # Safeguard mismatch lengths on network errors

    # FIXED: Updated tracking log to accurately reflect the 200-row scale
    if (index + 1) % 20 == 0:
        print(f"Progress: Processed {index + 1}/200 rows for all 6 models...")

    time.sleep(1) # Protect OpenRouter RPM (requests per minute) rate limits

# Save scores to master dataframe columns
for model_name, score_list in results.items():
    df_eval[f'score_{model_name}'] = score_list

print("EVALUATION COMPLETED! All automated judgments stored.")

RUNNING THE 6-MODEL BENCHMARK LOOP (200 TOTAL CASES)...
Progress: Processed 20/200 rows for all 6 models...
Progress: Processed 40/200 rows for all 6 models...
Progress: Processed 60/200 rows for all 6 models...
Progress: Processed 80/200 rows for all 6 models...
Progress: Processed 100/200 rows for all 6 models...
Progress: Processed 120/200 rows for all 6 models...
Progress: Processed 140/200 rows for all 6 models...
Progress: Processed 160/200 rows for all 6 models...
Progress: Processed 180/200 rows for all 6 models...
Progress: Processed 200/200 rows for all 6 models...
EVALUATION COMPLETED! All automated judgments stored.


## Generate Statistics

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score
from google.colab import drive

# Mount drive to save files permanently
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'

ground_truth = df_eval['ground_truth'].tolist()
final_report = []

for model_name in judges.keys():
    predictions = df_eval[f'score_{model_name}'].tolist()

    # Calculate Overall Accuracy against HaluEval Human Ground Truth
    accuracy = accuracy_score(ground_truth, predictions)

    # Calculate Cohen's Kappa Coefficient (Statistical reliability metric)
    kappa = cohen_kappa_score(ground_truth, predictions)

    final_report.append({
        "Model Evaluator": model_name,
        "Benchmark Accuracy (%)": round(accuracy * 100, 2),
        "Cohen's Kappa Coefficient": round(kappa, 3)
    })

# Convert to DataFrame, sort by accuracy, and display
metrics_df = pd.DataFrame(final_report).sort_values(by="Benchmark Accuracy (%)", ascending=False)
display(metrics_df)

# Export data logs directly to  Google Drive folder
metrics_df.to_csv(f"{DRIVE_PATH}6_JUDGES_METAEVAL_RESULTS.csv", index=False)
df_eval.to_csv(f"{DRIVE_PATH}6_JUDGES_RAW_LOGS.csv", index=False)

print("\n METRICS GENERATED SUCCESSFULLY!")
print(f"The final summary has been exported to drive: {DRIVE_PATH}6_JUDGES_METAEVAL_RESULTS.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Model Evaluator,Benchmark Accuracy (%),Cohen's Kappa Coefficient
3,CLAUDE-SONNET,87.5,0.75
4,LLAMA-3.1-70B,85.0,0.70
2,GEMINI-FLASH,83.5,0.67
0,GPT-4O-MINI,80.5,0.61
1,MISTRAL-LARGE,78.5,0.57
5,DEEPSEEK-CHAT,76.0,0.52



 METRICS GENERATED SUCCESSFULLY!
The final summary has been exported to drive: /content/drive/MyDrive/6_JUDGES_METAEVAL_RESULTS.csv


In [ ]:
display(metrics_df)

,Model Evaluator,Benchmark Accuracy (%),Cohen's Kappa Coefficient
3,CLAUDE-SONNET,87.5,0.75
4,LLAMA-3.1-70B,85.0,0.70
2,GEMINI-FLASH,83.5,0.67
0,GPT-4O-MINI,80.5,0.61
1,MISTRAL-LARGE,78.5,0.57
5,DEEPSEEK-CHAT,76.0,0.52
